# 03 - Retrieval Evaluation

Evaluate hit rate and MRR for keyword, vector, and hybrid search. Optimize keyword boost parameters.

In [ ]:
import json, os, sys, random
import numpy as np
sys.path.insert(0, "..")
from minsearch import Index
from app.search import keyword_search, vector_search, hybrid_search
from app.evaluation import hit_rate, mrr, evaluate_retrieval, optimize_boosts
from sentence_transformers import SentenceTransformer

In [ ]:
with open("../data/processed/fastapi/chunks.json") as f:
    chunks = json.load(f)
embeddings = np.load("../data/processed/fastapi/embeddings.npy")
gt_path = "../data/ground_truth.jsonl"
if os.path.exists(gt_path):
    with open(gt_path) as f:
        ground_truth = [json.loads(line) for line in f]
    print(f"Loaded {len(ground_truth)} ground truth pairs")
else:
    ground_truth = []
    print("No ground truth found, run 02-ground-truth.ipynb first")

In [ ]:
index = Index(text_fields=["title", "section", "content"], keyword_fields=["id", "doc_library"])
index.fit(chunks)
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Evaluate all three methods
methods = {
    "keyword": lambda q: keyword_search(q, index=index, num_results=5),
    "vector": lambda q: vector_search(q, embeddings=embeddings, chunks=chunks, model=model, num_results=5),
    "hybrid": lambda q: hybrid_search(q, method="hybrid", index=index, embeddings=embeddings, chunks=chunks, model=model, num_results=5),
}
results = {}
for name, fn in methods.items():
    metrics = evaluate_retrieval(ground_truth, fn)
    results[name] = metrics
    print(f"{name:>8}: HR={metrics['hit_rate']:.3f}, MRR={metrics['mrr']:.3f}")

In [ ]:
# Optimize keyword boost parameters
random.seed(42)
best_params = optimize_boosts(index, ground_truth, iterations=30)
print(f"Best boost params: {best_params}")
def search_optimized(q):
    return index.search(q, boost_dict=best_params, num_results=5)
opt_metrics = evaluate_retrieval(ground_truth, search_optimized)
print(f"Optimized keyword: HR={opt_metrics['hit_rate']:.3f}, MRR={opt_metrics['mrr']:.3f}")

In [ ]:
import pandas as pd
summary = pd.DataFrame(results).T
summary.columns = ["Hit Rate", "MRR"]
summary.loc["keyword_opt"] = [opt_metrics["hit_rate"], opt_metrics["mrr"]]
print("\nComparison:")
print(summary.to_string(float_format="%.3f"))